# Table 1: Best balanced static policy versus optimal static policy

This standalone notebook reproduces Table 1 of *Tight Guarantees for Static Policies in Vehicle Sharing Networks*. It compares the revenue of the best offered-balanced static policy with that of the optimal static policy in a finite-fleet vehicle-sharing network. It contains the numerical methods, instance generator, verification checks, and CSV export. No local Python modules or previously generated results are required.

## Network and demand model

The network has $N$ locations and a closed fleet of $C$ vehicles. It is a **complete directed network without self-loops**: $\mathcal A=\{(i,j):i,j\in\{1,\ldots,N\},\ i\ne j\}$, with $N(N-1)$ arcs. The arcs $(i,j)$ and $(j,i)$ have independently generated parameters. The topology is fixed for a given $N$; instances differ in their demand and service parameters.

For every arc $(i,j)$ in every instance, parameters are drawn independently as

$$a^{ij}\sim\mathrm{Uniform}[0.1,5],\qquad b^{ij}\sim\mathrm{Uniform}[0.5,10],\qquad \mu^{ij}\sim\mathrm{Uniform}[0.2,5].$$

The service rate $\mu^{ij}$ corresponds to mean travel time $1/\mu^{ij}$. The computation uses these means to evaluate stationary revenue; it does not simulate vehicle trajectories.

The two **demand functions** are specified below in terms of the offered demand rate $\lambda^{ij}$ and its price $p^{ij}(\lambda^{ij})$. Offered revenue on an arc is $r^{ij}(\lambda^{ij})=\lambda^{ij}p^{ij}(\lambda^{ij})$.

| Demand | Price as a function of offered demand rate | Offered revenue |
|---|---|---|
| Linear | $p^{ij}(\lambda^{ij})=b^{ij}-a^{ij}\lambda^{ij}$ | $r^{ij}(\lambda^{ij})=b^{ij}\lambda^{ij}-a^{ij}(\lambda^{ij})^2$ |
| Exponential | $p^{ij}(\lambda^{ij})=a^{ij}\log\!\left(\frac{b^{ij}}{a^{ij}\lambda^{ij}}\right)$ | $r^{ij}(\lambda^{ij})=a^{ij}\lambda^{ij}\log\!\left(\frac{b^{ij}}{a^{ij}\lambda^{ij}}\right)$ |

For both families, the maximum demand rate at price zero is $\Lambda^{ij}=b^{ij}/a^{ij}$, and $0\le\lambda^{ij}\le\Lambda^{ij}$. At $\lambda^{ij}=0$, revenue is defined by its continuous extension, $r^{ij}(0)=0$. A single demand family is used on all arcs of an instance.

The **optimal static policy** maximizes stationary revenue subject to vehicle-flow balance. The **offered-balanced policy** additionally requires outgoing and incoming offered demand rates to balance at every location:

$$\sum_{j:j\ne i}\lambda^{ij}=\sum_{k:k\ne i}\lambda^{ki},\qquad i=1,\ldots,N.$$

## Installation and execution

Use Python 3.12 with a Jupyter-compatible notebook interface. Install the numerical dependencies in the notebook's Python environment:

```python
%pip install numpy scipy cvxpy clarabel
```

Restart the kernel after installation, then select **Restart Kernel and Run All**. All code cells must be executed in order. For a quick execution check, set `NUM_INSTANCES = 1`; this runs 24 instances with the other defaults. The default experiment uses 100 instances per setting, for 2,400 instances in total, and runs sequentially.

The implementation was tested with Python 3.12.13, NumPy 2.5.1, SciPy 1.18.0, CVXPY 1.9.2, and CLARABEL 0.11.1. Solver statuses and last-digit results can vary across package versions and platforms; every run applies the acceptance checks described below.

## Experiment settings and reproducibility

| Setting | Default | Meaning |
|---|---|---|
| `SEED` | `77` | Seed for NumPy's default random-number generator |
| `NUM_INSTANCES` | `100` | Independent instances per $(N,C)$ and demand family |
| `N_VALUES` | `(5, 10, 15, 20)` | Number of locations |
| `C_VALUES` | `(5, 20, 40)` | Number of vehicles |
| `DEMAND_FAMILIES` | `("Linear", "Exponential")` | Demand families |
| `RELATIVE_GAP` | `1e-6` | Maximum relative revenue upper/lower bound gap |
| `FEASIBILITY_TOL` | `1e-8` | Maximum absolute constraint residual |
| `OUTPUT` | `output/table1` | Output directory relative to the kernel's working directory |

Random draws follow the order demand family, $N$, $C$, instance, then all arc values of $a$, $b$, and $\mu$. Arcs are ordered by origin and then destination, omitting diagonal entries. Parameter draws are independent across demand families, network settings, and instances; different fleet sizes do not share the same sampled network parameters.

Changing the number or order of settings or instances changes later draws, even with the same seed. Consequently, a run with one instance per setting does not reproduce the first instance of every group in the default experiment. After changing settings, restart the kernel and run all cells.

## Numerical methods and accuracy

CVXPY constructs the optimization models, and **CLARABEL** solves the internal conic problems. The optimal static policy is computed using supporting-plane relaxations of the convex constraint $Q_C\le1$. The normalizer ratio $Q_C$ and its gradient are evaluated by a scaled recursion. Feasible policies provide revenue lower bounds, and the relaxation multipliers give numerical global upper bounds.

The offered-balanced policy is computed by solving the flow problem for trial workload limits $\Delta$ and applying **SciPy's bounded scalar search** to the resulting finite-fleet revenue. The initial workload range is $[0,\sum_{(i,j)\in\mathcal A}\Lambda^{ij}/\mu^{ij}]$; the search trims this range using the workload of the flow optimum with only flow-balance and rate-cap constraints. Revenue evaluation uses the returned policy's actual workload. A direct common-weight formulation supplies an independent upper bound to check the search result, with additional search refinement when needed.

Each policy is accepted only if its revenue and bounds are finite, the bounds are correctly ordered, the relative gap `(upper_bound - revenue) / revenue` is at most `RELATIVE_GAP`, and the maximum constraint residual is at most `FEASIBILITY_TOL`. The direct offered-balanced upper-bound calculation uses the tighter gap `RELATIVE_GAP / 3`. `repair_policy` constructs the feasible point used for the revenue lower bound before these checks.

CLARABEL's internal gap and feasibility tolerances are `1e-10`. A solver exception triggers a retry at `1e-9` with a shorter step; the final acceptance criteria remain the same. Supporting-plane calculations first use `max_cuts=64` and retry with `128` after a numerical convergence failure. An internal `optimal_inaccurate` status can be accepted only when the independent final checks pass. Solver warnings and retry reasons are recorded by stage. An unresolved failure raises an exception and stops the experiment.

These are floating-point numerical accuracy checks based on the convexity of $Q_C$ established in Theorem 3 of the accompanying manuscript, *Tight Guarantees for Static Policies in Vehicle Sharing Networks*. They are not interval-arithmetic certificates. The `certified` field in the returned diagnostics indicates that these numerical checks passed.

## Results and output files

The reported ratio is **offered-balanced revenue / optimal static revenue**, using the feasible revenues returned by the two policy functions. Its numerical lower bound divides the offered-balanced feasible revenue by the optimal static upper bound; its numerical upper bound divides the offered-balanced upper bound by the optimal static feasible revenue. The point estimate and both bounds are stored separately; ratios are not clipped at one.

| File | Contents |
|---|---|
| `instance_results.csv` | Per-instance revenues, ratio estimates and bounds, maximum gaps and residuals, and solver statuses and warnings labeled by stage |
| `summary.csv` | Average and minimum ratios for each $(N,C)$ and demand family; these are the values reported in Table 1 |

The minimum is the smallest ratio among the sampled instances in a setting, not a theoretical worst-case guarantee.

CSV files are written after the experiment cell completes. This notebook has no checkpoint or resume mechanism: rerunning starts from the seed and replaces the output files upon completion. Set a different `OUTPUT` directory to retain separate experiments. The `rows` and `summary` variables remain available for analysis.

## Code organization

1. **Basic calculations and shared numerical tools:** network balance, demand revenue, $Q_C$, solver settings, feasible-policy construction, and residual checks.
2. **Optimal static policy:** `solve_optimal_static_policy(...)` returns `(revenue, info)`. The shared conic implementation is `_solve_static_revenue`.
3. **Offered-balanced policy:** `solve_offered_balanced_policy(...)` returns `(revenue, info)` from workload search. It computes the direct verification bound internally; `info['direct_revenue']` and `info['direct_info']` retain that calculation's results. Both policy functions return the policy vectors as `info['omega']` and `info['z']`.
4. **Table 1 experiment:** generate inputs, evaluate both policies on each instance, and save the detailed diagnostics and Table 1 summary.


## 1. Basic calculations and shared numerical tools

Imports and experiment settings are followed by functions used by both policies.


In [9]:
# Standalone notebook. Run all cells in a Python kernel with the dependencies below.
# If needed: %pip install numpy scipy cvxpy clarabel
import os
import csv
import warnings
from pathlib import Path
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import cvxpy as cp
from scipy.special import xlogy
from scipy.optimize import minimize_scalar

assert "CLARABEL" in cp.installed_solvers(), "Install clarabel in this notebook's kernel."

In [10]:
SEED = 77
NUM_INSTANCES = 100  # Instances per setting and demand family; use 1 for a quick execution check.
N_VALUES = (5, 10, 15, 20)
C_VALUES = (5, 20, 40)
DEMAND_FAMILIES = ("Linear", "Exponential")
RELATIVE_GAP = 1e-6
FEASIBILITY_TOL = 1e-8
OUTPUT = Path("output/table1")

In [11]:
# Shared by Parts 2 and 3: solver settings, network, Q_C, revenue, and feasibility.
def _solve_conic(problem):
    # CVXPY builds the conic model; CLARABEL performs the numerical solve.
    # These internal tolerances differ from the final revenue-gap checks.
    try:
        return problem.solve(solver='CLARABEL', tol_gap_abs=1e-10, tol_gap_rel=1e-10,
                             tol_feas=1e-10, max_iter=200, warm_start=True)
    except cp.error.SolverError as exc:
        # Restart with a shorter interior-point step after a numerical exception.
        # The caller still enforces the same final gap and feasibility tolerances.
        warnings.warn(f'CLARABEL exception; retry with max_step_fraction=0.9, tolerance=1e-9: {exc}', RuntimeWarning)
        return problem.solve(solver='CLARABEL', tol_gap_abs=1e-9, tol_gap_rel=1e-9,
                             tol_feas=1e-9, max_iter=300, max_step_fraction=.9,
                             warm_start=False)


def network(N):
    # Arc arrays list every ordered pair (i,j), i != j, in row-major order.
    origins, destinations = np.where(~np.eye(N, dtype=bool))
    # balance @ z is outgoing minus incoming flow at each retained location.
    # The last row follows from the others because total net flow is zero.
    balance = np.zeros((N - 1, len(origins)))
    for i in range(N - 1):
        balance[i] = (origins == i).astype(float) - (destinations == i)
    return origins, destinations, balance


def q_and_gradient(omega, rho, C):
    # Inputs: N location weights, total travel load rho=sum(z/mu), fleet size C.
    # Returns Q_C, its N weight derivatives, and its derivative w.r.t. rho.
    omega = np.asarray(omega, dtype=float)
    # Work with weights <= 1 and scaled travel load <= C to control magnitudes.
    # The small floor prevents division by zero; normalization is numerical only.
    scale = max(float(np.max(omega)), float(rho) / C, 1e-100)
    weights = omega / scale
    load = rho / scale
    # Initially G[k]=load**k/k!, the aggregated infinite-server contribution.
    # Successive multiplication avoids forming powers and factorials separately.
    G = np.ones(C + 1)
    for k in range(1, C + 1):
        G[k] = G[k - 1] * load / k
    # Add one location at a time. Ascending k uses the UPDATED G[k-1].
    for weight in weights:
        for k in range(1, C + 1):
            G[k] += weight * G[k - 1]
    Q = G[C] / G[C - 1]
    # H adds one extra copy of each location weight, in parallel.
    # At the end H[i]=dG_C/dweights[i], previous[i]=dG_(C-1)/dweights[i].
    H = np.ones(len(weights))
    previous = np.zeros(len(weights))
    for k in range(1, C):
        previous = H.copy()
        H = G[k] + weights * H
    # Quotient rule, written without a squared normalizer denominator.
    domega = (H - Q * previous) / G[C - 1]
    # dG_k/dload=G_(k-1); for C=1, dG_0/dload=0.
    drho = 1.0 if C == 1 else 1.0 - Q * G[C - 2] / G[C - 1]
    # Q_C has degree one, so restore its scale. Its gradient has degree zero.
    return Q * scale, domega, drho


def arc_revenue(lam, a, b, family):
    # Per-arc offered revenue r(lambda)=lambda*p(lambda), before availability.
    if family == "Linear":
        return b * lam - a * lam**2
    # xlogy gives the continuous value 0 at lambda=0, avoiding 0*log(0).
    return a * (xlogy(lam, b / a) - xlogy(lam, lam))


def best_arc_rates(dual_cost, a, b, family):
    # Maximize r(lambda)-dual_cost*lambda on [0,b/a], separately for each arc.
    # dual_cost comes from dual multipliers; this maximization builds the upper bound.
    if family == "Linear":
        return np.clip((b - dual_cost) / (2 * a), 0, b / a)
    # The upper exponent 0 imposes lambda<=b/a; -745 handles the float limit.
    return (b / a) * np.exp(np.clip(-1 - dual_cost / a, -745, 0))


def repair_policy(omega, z, N, C, a, b, mu, offered_balance):
    # Convert a relaxation candidate into a numerically feasible point of (14).
    # Rates are recovered as lambda=z/omega[origins]; inputs are copied.
    origins, destinations, _ = network(N)
    z = np.maximum(np.asarray(z, float), 0).copy()
    # Positive weights avoid division by zero when recovering rates.
    omega = np.maximum(np.asarray(omega, float), 1e-14).copy()
    net = np.bincount(origins, weights=z, minlength=N) - np.bincount(destinations, weights=z, minlength=N)
    # net<0 needs more outgoing flow; net>0 needs more incoming flow.
    # A complete directed network always has the required arc i -> j.
    lookup = np.full((N, N), -1, int)
    lookup[origins, destinations] = np.arange(len(z))
    for i in np.flatnonzero(net < 0):
        for j in np.flatnonzero(net > 0):
            amount = min(-net[i], net[j])
            if amount > 0:
                z[lookup[i, j]] += amount
                net[i] += amount
                net[j] -= amount
    # Raise each origin weight enough that z_ij <= (b_ij/a_ij)*omega_i.
    np.maximum.at(omega, origins, z / (b / a))
    if offered_balance:
        # A shared maximum preserves all demand bounds and offered balance.
        omega[:] = omega.max()
    Q, _, _ = q_and_gradient(omega, np.sum(z / mu), C)
    # Homogeneity places Q_C just below 1. Common scaling preserves flow
    # balance, demand bounds, and the recovered rates z/omega[origins].
    # The resulting perspective objective is a conservative revenue lower bound.
    omega /= Q * (1 + 2e-12)
    z /= Q * (1 + 2e-12)
    return omega, z


def diagnostics(omega, z, N, C, a, b, mu, offered_balance):
    # Check every location, including the omitted balance row, after repair.
    # Each inequality residual is zero when satisfied, positive when violated.
    origins, destinations, _ = network(N)
    net = np.bincount(origins, weights=z, minlength=N) - np.bincount(destinations, weights=z, minlength=N)
    Q, _, _ = q_and_gradient(omega, np.sum(z / mu), C)
    result = dict(
        flow_balance_violation=float(np.max(np.abs(net))),
        offered_balance_violation=float(np.ptp(omega)) if offered_balance else 0.,
        demand_bound_violation=float(max(0., np.max(z - b / a * omega[origins]))),
        nonnegativity_violation=float(max(0., -omega.min(), -z.min())),
        Q_C_value=float(Q), Q_C_violation=float(max(0, Q - 1)),
    )
    result['max_constraint_violation'] = max(v for k, v in result.items() if k.endswith('violation'))
    # NaN/inf must be rejected explicitly, independently of residual comparisons.
    result['finite'] = bool(np.all(np.isfinite(omega)) and np.all(np.isfinite(z)) and np.isfinite(Q))
    return result

## 2. Optimal static policy

The shared conic model comes first; `solve_optimal_static_policy` is the entry used by the experiment.


In [12]:
# Shared implementation: optimal static policy uses N weights; the offered-balanced
# verification in Part 3 uses one common weight. Keep this numerical model in one place.
def _solve_static_revenue(N, C, a, b, mu, demand_family, offered_balance=False,
                         relative_gap=1e-6, feasibility_tol=1e-8, max_cuts=64):
    # Solve (14), optionally restricted to common weights, using outer approximation.
    # Returns a feasible revenue lower bound and diagnostics, bounds, omega, and z.
    a, b, mu = (np.asarray(x, dtype=float) for x in (a, b, mu))
    # This notebook models complete directed networks without self-loops.
    if not isinstance(N, (int, np.integer)) or N < 2 or not isinstance(C, (int, np.integer)) or C < 1:
        raise ValueError('Require integer N >= 2 and integer C >= 1.')
    if demand_family not in ('Linear', 'Exponential'):
        raise ValueError('demand_family must be Linear or Exponential.')
    if any(x.shape != (N * (N - 1),) or not np.all(np.isfinite(x) & (x > 0)) for x in (a, b, mu)):
        raise ValueError('a, b, mu must each contain N*(N-1) finite, positive arc values.')
    if not np.isfinite(relative_gap) or relative_gap <= 0 or not np.isfinite(feasibility_tol) or feasibility_tol <= 0:
        raise ValueError('Require finite, positive gap and feasibility tolerances.')
    if not isinstance(max_cuts, (int, np.integer)) or max_cuts < 2:
        raise ValueError('max_cuts must be an integer >= 2.')
    
    origins, destinations, balance = network(N)
    arcs = len(origins)
    # One shared variable eliminates the N-1 common-weight equalities.
    nw = 1 if offered_balance else N
    w = cp.Variable(nw, nonneg=True)
    z = cp.Variable(arcs, nonneg=True)

    # Repeat the appropriate origin weight for every arc in the objective/bounds.
    omega_arcs = w[0] * np.ones(arcs) if offered_balance else w[origins]

    # Reserve cut rows so coefficients can change without rebuilding the model.
    cuts = cp.Parameter((max_cuts, nw + arcs), value=np.zeros((max_cuts, nw + arcs)))
    rhs = np.ones(max_cuts)
    # A positive symmetric circulation supplies a revenue scale, not a restriction.
    rates_matrix = np.zeros((N, N))
    rates_matrix[origins, destinations] = b / a
    ray = .08 * np.minimum(rates_matrix, rates_matrix.T)[origins, destinations]
    ow, oz = repair_policy(np.ones(N), ray, N, C, a, b, mu, offered_balance)

    scale = float(np.sum(ow[origins] * arc_revenue(oz / ow[origins], a, b, demand_family)))
    # Dividing the objective by a positive constant improves numerical scaling.
    scale = max(scale, 1.)

    eq = balance @ z == 0
    supporting = cuts @ cp.hstack([w, z]) <= rhs
    # w<=1 is valid because Q_C>=max(omega); cuts relax Q_C<=1.
    constraints = [eq, z <= cp.multiply(b / a, omega_arcs), w <= 1, supporting]
    if demand_family == 'Linear':
        t = cp.Variable(arcs)
        # Each cone enforces t_ij >= (a_ij/scale)*z_ij**2/omega_i.
        # Subtracting sum(t) represents the concave quadratic perspective.
        constraints += [cp.SOC(t + omega_arcs, cp.vstack([2 * cp.multiply(np.sqrt(a / scale), z), t - omega_arcs]), axis=0)]
        objective = b @ z / scale - cp.sum(t)
    else:
        # -a*rel_entr(z,Lambda*omega)=a*z*log(Lambda*omega/z).
        # CVXPY represents this expression with exponential cones.
        objective = -cp.sum(cp.multiply(a / scale, cp.rel_entr(z, cp.multiply(b / a, omega_arcs))))
    problem = cp.Problem(cp.Maximize(objective), constraints)

    matrix = np.zeros((max_cuts, nw + arcs))
    # Start with sum(z/mu)<=C, implied by Q_C>=sum(z/mu)/C.
    matrix[0, nw:] = 1 / (C * mu)
    # Keep the largest feasible objective and smallest valid upper bound seen.
    lower, upper = 0., np.inf
    best = None
    statuses, solver_warnings = [], []
    for iteration in range(1, max_cuts):
        cuts.value = matrix
        # Keep inner warnings for reporting; acceptance is decided by bounds below.
        with warnings.catch_warnings(record=True) as caught:
            _solve_conic(problem)

        solver_warnings.extend(str(item.message) for item in caught)
        statuses.append(problem.status)
        if (z.value is None or w.value is None or supporting.dual_value is None or eq.dual_value is None
                or any(not np.all(np.isfinite(x)) for x in
                       (z.value, w.value, supporting.dual_value, eq.dual_value))):
            raise RuntimeError(f'Conic relaxation lacks finite primal/dual values: {problem.status}')

        # Restore multipliers to original revenue units. Inequality multipliers
        # must be nonnegative; equality multipliers nu may have either sign.
        gamma = np.maximum(np.asarray(supporting.dual_value), 0) * scale
        nu = np.asarray(eq.dual_value) * scale
        # In the Lagrangian, dual_cost is the coefficient charged to each arc flow.
        # Maximize over lambda=z/omega, subject to 0<=lambda<=b/a.
        dual_cost = balance.T @ nu + matrix[:, nw:].T @ gamma
        lam = best_arc_rates(dual_cost, a, b, demand_family)
        arc_max = arc_revenue(lam, a, b, demand_family) - dual_cost * lam
        # The remaining coefficient charged to each location weight.
        idle_cost = matrix[:, :nw].T @ gamma
        if offered_balance:
            # One common weight lies in [0,1]: sum all arc contributions
            # before taking the positive part. gamma.sum() uses cut RHS=1.
            dual_bound = gamma.sum() + max(0., arc_max.sum() - idle_cost[0])
        else:
            # Independent weights in [0,1]: group by origin, then take each
            # positive part. This explicitly maximizes the Lagrangian.
            dual_bound = gamma.sum() + np.maximum(0, np.bincount(origins, weights=arc_max, minlength=N) - idle_cost).sum()
        if not np.isfinite(dual_bound):
            raise RuntimeError('Nonfinite revenue upper bound.')
        upper = min(upper, float(dual_bound))
        raw_omega = np.repeat(w.value[0], N) if offered_balance else w.value
        # A relaxation candidate can violate Q_C<=1. Repair before recording revenue.
        omega, flows = repair_policy(raw_omega, z.value, N, C, a, b, mu, offered_balance)
        revenue = float(np.sum(omega[origins] * arc_revenue(flows / omega[origins], a, b, demand_family)))
        if not np.isfinite(revenue) or revenue <= 0:
            raise RuntimeError('Conic candidate did not produce finite, positive revenue.')
        if revenue > lower:
            lower = revenue
            best = (omega.copy(), flows.copy())
        # Never turn a reversed interval into a successful zero gap.
        if upper < lower:
            raise RuntimeError(f'Inconsistent global bounds: {lower=} {upper=}')
        # Accept only when relative optimality gap AND repaired feasibility pass.
        # These are floating-point bounds under Theorem 3, not interval certificates.
        if (upper - lower) / lower <= relative_gap:
            check = diagnostics(*best, N, C, a, b, mu, offered_balance)
            if not check['finite'] or check['max_constraint_violation'] > feasibility_tol:
                raise RuntimeError(f'Policy repair failed: {check}')
            # solver_success/message describe the LAST inner solve; certified
            # describes the independent numerical checks on the saved best point.
            return lower, dict(**check, revenue=lower, upper_bound=upper,
                relative_gap=(upper - lower) / lower, certified=True,
                solver_success=problem.status == cp.OPTIMAL, message=problem.status,
                iterations=iteration, attempts=len(statuses), failed_attempts=sum(s != cp.OPTIMAL for s in statuses),
                solver_warnings=solver_warnings,
                omega=best[0], z=best[1])
        # Convexity gives a supporting plane; degree-one homogeneity removes
        # its intercept, so the new inequality has RHS 1.
        _, dw, dr = q_and_gradient(np.maximum(raw_omega, 0), np.sum(np.maximum(z.value, 0) / mu), C)
        # Chain rule: a shared weight has derivative sum(dw); each flow
        # has derivative dr/mu because rho=sum(z/mu).
        matrix[iteration, :nw] = dw.sum() if offered_balance else dw
        matrix[iteration, nw:] = dr / mu
    raise RuntimeError(f'No verified convergence after {max_cuts - 1} cuts; gap={(upper - lower)/lower:.3e}')

def solve_optimal_static_policy(N, C, a, b, mu, demand_family,
                                relative_gap=1e-6, feasibility_tol=1e-8):
    # Public entry: return the policy revenue and its numerical accuracy checks.
    # Retry with more supporting planes if the initial budget is insufficient.
    retry_messages = []
    for max_cuts in (64, 128):
        try:
            revenue, info = _solve_static_revenue(N, C, a, b, mu, demand_family,
                relative_gap=relative_gap, feasibility_tol=feasibility_tol, max_cuts=max_cuts)
            info['solver_warnings'] = retry_messages + info['solver_warnings']
            return revenue, info
        except RuntimeError as error:
            if max_cuts == 128:
                raise
            retry_messages.append(f'Retrying optimal static policy with 128 cuts after: {error}')


## 3. Offered-balanced policy

One entry contains the direct upper-bound calculation, workload search over the flow problem (8), and final checks.


In [13]:
def solve_offered_balanced_policy(N, C, a, b, mu, demand_family,
                                  relative_gap=1e-6, feasibility_tol=1e-8):
    # Public entry: the returned policy comes from line search over the flow problem (8).
    # Direct common-weight optimization supplies the independent upper bound only.
    # Both calculations and their checks are handled here, not in the experiment loop.
    a, b, mu = map(np.asarray, (a, b, mu))

    def search_workload(upper_bound):
        # Solve (8) over Delta and check against the direct common-weight upper bound.
        if not np.isfinite(upper_bound) or upper_bound <= 0:
            raise RuntimeError('Flow verification requires a finite, positive upper bound.')
        origins, _, balance = network(N)
        Lambda = b / a
        # Here lam contains OFFERED rates; Delta bounds their total travel load.
        lam = cp.Variable(len(a), nonneg=True)
        Delta = cp.Parameter(nonneg=True)
        # Unconstrained per-arc revenues supply a fixed objective scale.
        scale = max(1., float(arc_revenue(best_arc_rates(np.zeros(len(a)), a, b, demand_family), a, b, demand_family).sum()))
        # Offered-flow conservation, rate caps, and the workload inequality in (8).
        constraints = [balance @ lam == 0, lam <= Lambda, (1 / mu) @ lam <= Delta]
        if demand_family == 'Linear':
            objective = (b @ lam - cp.sum_squares(cp.multiply(np.sqrt(a), lam))) / scale
        else:
            objective = -cp.sum(cp.multiply(a / scale, cp.rel_entr(lam, Lambda)))
        # Reuse this parameterized model for every trial Delta.
        problem = cp.Problem(cp.Maximize(objective), constraints)
        evaluations, statuses, solver_warnings = [], [], []
        best_revenue, best = 0., None

        def evaluate(delta):
            # One outer-search evaluation: solve (8), then evaluate finite-fleet revenue.
            nonlocal best_revenue, best
            # Zero workload forces zero rates and therefore zero revenue.
            if delta <= 0:
                return 0.
            Delta.value = float(delta)
            with warnings.catch_warnings(record=True) as caught:
                _solve_conic(problem)
            solver_warnings.extend(str(item.message) for item in caught)
            statuses.append(problem.status)
            if lam.value is None or not np.all(np.isfinite(lam.value)):
                raise RuntimeError(f'Flow problem failed at {delta=}: {problem.status}')
            # Balanced rates admit common weights: begin with omega=1, z=lam.
            # Repair and normalize using Q_C, thereby accounting for availability.
            omega, z = repair_policy(np.ones(N), lam.value, N, C, a, b, mu, True)
            revenue = float(np.sum(omega[origins] * arc_revenue(z / omega[origins], a, b, demand_family)))
            if not np.isfinite(revenue) or revenue <= 0:
                raise RuntimeError(f'Flow candidate has invalid revenue at {delta=}.')
            # Use the repaired policy's actual load; it need not equal trial Delta.
            actual_load = float(np.sum(z / omega[origins] / mu))
            evaluations.append((float(delta), revenue, actual_load))
            if revenue > best_revenue:
                best_revenue, best = revenue, (omega, z, delta, actual_load)
            # SciPy minimizes, so negate the revenue; save the best policy separately.
            return -revenue

        # Above the workload of the unconstrained flow optimum, (8) is unchanged.
        # At this large Delta the rate caps already enforce the workload bound.
        # The resulting optimum's workload trims the redundant search interval.
        evaluate(float(np.sum(Lambda / mu)))
        search_upper = min(float(np.sum(Lambda / mu)), evaluations[-1][2] * (1 + 1e-8))
        # Bounded Brent search proposes a candidate; it is not a global certificate.
        minimize_scalar(evaluate, bounds=(0., search_upper), method='bounded',
                        options={'xatol': max(1e-8, search_upper * 1e-8), 'maxiter': 80})
        if (upper_bound - best_revenue) / best_revenue > relative_gap:
            # If the candidate is not close enough to the independent upper bound,
            # sample more workloads and refine brackets around sampled local peaks.
            grid = np.linspace(0, search_upper, 17)
            values = [0.] + [-evaluate(x) for x in grid[1:]]
            for k in range(1, len(grid) - 1):
                if values[k] >= max(values[k - 1], values[k + 1]):
                    minimize_scalar(evaluate, bounds=(grid[k - 1], grid[k + 1]), method='bounded',
                                    options={'xatol': max(1e-9, search_upper * 1e-9)})
        # Validate the best policy encountered, not necessarily the last trial point.
        check = diagnostics(best[0], best[1], N, C, a, b, mu, True)
        gap = (upper_bound - best_revenue) / best_revenue
        if not check['finite'] or check['max_constraint_violation'] > feasibility_tol:
            raise RuntimeError(f'Flow policy feasibility failed: {check}')
        # Reject nonfinite or reversed bounds; never clip a negative gap to zero.
        if not np.isfinite(gap) or gap > relative_gap or gap < 0:
            raise RuntimeError(f'Workload search did not reach global bound: {gap=}')
        # Delta/actual_load/omega/z belong to the best point; message is the last inner status.
        return best_revenue, dict(**check, revenue=best_revenue, upper_bound=upper_bound,
            relative_gap=gap, certified=True,
            solver_success=statuses[-1] == cp.OPTIMAL, message=statuses[-1],
            iterations=len(evaluations), attempts=len(statuses), failed_attempts=sum(s != cp.OPTIMAL for s in statuses),
            solver_warnings=solver_warnings,
            Delta=float(best[2]), actual_load=best[3], omega=best[0], z=best[1])
    # Tighten the direct gap so the flow search has room within relative_gap.
    # If verification fails, retry the offered-balanced calculation with more cuts.
    retry_messages = []
    for max_cuts in (64, 128):
        try:
            direct_revenue, direct_info = _solve_static_revenue(N, C, a, b, mu, demand_family,
                offered_balance=True, relative_gap=relative_gap / 3,
                feasibility_tol=feasibility_tol, max_cuts=max_cuts)
            revenue, info = search_workload(direct_info['upper_bound'])
            # Keep the direct solve's diagnostics separate from the flow solver's.
            info['solver_warnings'] = retry_messages + info['solver_warnings']
            info['direct_revenue'] = direct_revenue
            info['direct_info'] = direct_info
            return revenue, info
        except RuntimeError as error:
            if max_cuts == 128:
                raise
            retry_messages.append(f'Retrying offered-balanced policy with 128 cuts after: {error}')


## 4. Table 1 experiment

Call the two policies on identical inputs, save the detailed diagnostics, and export the minimum and average ratios reported in Table 1.


In [14]:
# Run independent instances sequentially in a deterministic random draw order.
# Changing NUM_INSTANCES changes the draws for later settings. No previous results are loaded.
generator = np.random.default_rng(SEED)
rows, summary = [], []
for demand_family in DEMAND_FAMILIES:
    for N in N_VALUES:
        for C in C_VALUES:
            ratios = []
            for instance in range(1, NUM_INSTANCES + 1):
                a = generator.uniform(0.1, 5.0, N * (N - 1))
                b = generator.uniform(0.5, 10.0, N * (N - 1))
                mu = generator.uniform(0.2, 5.0, N * (N - 1))
                try:
                    static, static_info = solve_optimal_static_policy(N, C, a, b, mu, demand_family,
                        relative_gap=RELATIVE_GAP, feasibility_tol=FEASIBILITY_TOL)
                    flow, flow_info = solve_offered_balanced_policy(N, C, a, b, mu, demand_family,
                        relative_gap=RELATIVE_GAP, feasibility_tol=FEASIBILITY_TOL)
                except Exception as error:
                    raise RuntimeError(f"{demand_family}, N={N}, C={C}, instance={instance}") from error

                # The offered-balanced entry also retains its direct verification result.
                balanced = flow_info['direct_revenue']
                balanced_info = flow_info['direct_info']
                ratio = flow / static
                ratio_lower = flow / static_info["upper_bound"]
                ratio_upper = balanced_info["upper_bound"] / static
                if (not np.all(np.isfinite([ratio_lower, ratio, ratio_upper]))
                        or not 0 <= ratio_lower <= ratio <= ratio_upper
                        or ratio_lower > 1 + 1e-8 or ratio > 1 + 2 * RELATIVE_GAP):
                    raise RuntimeError(f'Revenue ratio interval failed: {demand_family}, N={N}, C={C}, instance={instance}.')
                infos = (static_info, balanced_info, flow_info)
                stages = ('optimal static', 'offered-balanced direct', 'offered-balanced flow')
                rows.append(dict(demand_family=demand_family, N=N, C=C, instance=instance,
                    optimal_static_revenue=static, offered_balanced_revenue=flow,
                    offered_balanced_direct_revenue=balanced, revenue_ratio=ratio,
                    ratio_lower=ratio_lower, ratio_upper=ratio_upper,
                    max_gap=max(info["relative_gap"] for info in infos),
                    max_violation=max(info["max_constraint_violation"] for info in infos),
                    inner_statuses=" | ".join(f'{stage}: {info["message"]}' for stage, info in zip(stages, infos)),
                    warning_messages=" | ".join(f'{stage}: {message}' for stage, info in zip(stages, infos)
                                                for message in info["solver_warnings"])))
                ratios.append(ratio)
            summary.append(dict(demand_family=demand_family, N=N, C=C,
                minimum_ratio=min(ratios), average_ratio=float(np.mean(ratios))))
            print(f"{demand_family}, N={N}, C={C}: minimum={min(ratios):.6f}, average={np.mean(ratios):.6f}", flush=True)

# Preserve the acceptance checks and solver messages in the CSV; no failed instances are dropped.
OUTPUT.mkdir(parents=True, exist_ok=True)
for filename, records in [("instance_results.csv", rows), ("summary.csv", summary)]:
    with (OUTPUT / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
print(f"Verified {len(rows)} instances; max gap={max(r['max_gap'] for r in rows):.3e}; "
      f"max residual={max(r['max_violation'] for r in rows):.3e}")

Linear, N=5, C=5: minimum=0.806625, average=0.947282
Linear, N=5, C=20: minimum=0.920191, average=0.979035
Linear, N=5, C=40: minimum=0.957697, average=0.986828
Linear, N=10, C=5: minimum=0.710826, average=0.942893
Linear, N=10, C=20: minimum=0.928215, average=0.974252
Linear, N=10, C=40: minimum=0.949352, average=0.983003
Linear, N=15, C=5: minimum=0.801494, average=0.947454
Linear, N=15, C=20: minimum=0.880989, average=0.965919
Linear, N=15, C=40: minimum=0.955886, average=0.980763
Linear, N=20, C=5: minimum=0.825538, average=0.947971
Linear, N=20, C=20: minimum=0.893536, average=0.967626
Linear, N=20, C=40: minimum=0.951905, average=0.979338
Exponential, N=5, C=5: minimum=0.893019, average=0.970409
Exponential, N=5, C=20: minimum=0.969651, average=0.990839
Exponential, N=5, C=40: minimum=0.988978, average=0.995480
Exponential, N=10, C=5: minimum=0.946998, average=0.977649
Exponential, N=10, C=20: minimum=0.974010, average=0.991292
Exponential, N=10, C=40: minimum=0.990223, average=0